# 01. Color Quantization

목표는 원본 이미지를 사람이 색칠하기 쉬운 제한된 색상 수로 단순화하는 것입니다.

비교 알고리즘:

- K-Means Clustering: 이미지 픽셀을 K개의 대표 색상 군집으로 묶습니다.
- Posterization: RGB 채널을 균등 구간으로 나누는 빠른 단순화 방식입니다.
- Median Cut: 색상 분포가 넓은 축을 반복 분할해 대표 팔레트를 만듭니다.

최종 기본 알고리즘은 색상 유지 품질과 발표 설명 용이성을 고려해 K-Means로 선택합니다.

In [ ]:
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np

# 공통 함수가 들어 있는 src 폴더를 import 경로에 추가합니다.
sys.path.append("src")
from coloringbook_utils import *

# 결과 저장 폴더(outputs)와 샘플 입력 폴더(data)를 생성합니다.
ensure_dirs()
plt.rcParams["figure.dpi"] = 120

# 직접 사용할 이미지가 있으면 여기에 경로를 넣으세요.
# 예: IMAGE_PATH = "data/my_image.jpg"
IMAGE_PATH = None

# IMAGE_PATH가 None이면 발표/실험용 샘플 이미지가 자동 생성됩니다.
image = load_image(IMAGE_PATH)
show_images([("Original Image", image)], cols=1, figsize=(5, 5))

## K-Means, Posterization, Median Cut 실행

아래 셀에서 원하는 색상 개수 `K`를 바꿀 수 있습니다.

In [ ]:
# K는 최종 팔레트에 남길 대표 색상 개수입니다.
# K가 작으면 색칠은 쉬워지고, K가 크면 원본 색상 보존이 좋아집니다.
K = 10

# K-Means: 픽셀을 K개 군집으로 묶고 각 군집 중심색으로 이미지를 치환합니다.
(kmeans_img, kmeans_palette), kmeans_time = timed_call(kmeans_quantization, image, K)

# Posterization: RGB 채널을 일정한 구간으로 나눠 빠르게 색상을 줄입니다.
(poster_img, poster_palette), poster_time = timed_call(posterization, image, K)

# Median Cut: 색상 분포가 넓은 축을 반복 분할해 대표 색상표를 만듭니다.
(median_img, median_palette), median_time = timed_call(median_cut_quantization, image, K)

# 각 알고리즘 결과를 파일로 저장해 보고서/발표 자료에 바로 사용할 수 있게 합니다.
save_image_rgb("outputs/01_kmeans.png", kmeans_img)
save_image_rgb("outputs/01_posterization.png", poster_img)
save_image_rgb("outputs/01_median_cut.png", median_img)

show_images([
    ("Original", image),
    (f"K-Means K={K} ({kmeans_time:.3f}s)", kmeans_img),
    (f"Posterization K≈{K} ({poster_time:.3f}s)", poster_img),
    (f"Median Cut K={K} ({median_time:.3f}s)", median_img),
], cols=2, figsize=(11, 8), save_path="outputs/01_quantization_compare.png")

## RGB 색상표

색상표는 최종 컬러링북 번호와 연결되는 색상 안내표로 사용할 수 있습니다.

In [ ]:
plot_palette(kmeans_palette, "K-Means RGB Palette", "outputs/01_kmeans_palette.png")
plot_palette(poster_palette, "Posterization RGB Palette", "outputs/01_poster_palette.png")
plot_palette(median_palette, "Median Cut RGB Palette", "outputs/01_median_palette.png")

## 5색, 10색, 20색 비교

색상 개수가 늘어나면 원본 보존력은 좋아지지만, 색칠해야 할 영역과 인지 복잡도도 함께 증가합니다.

In [ ]:
# 색상 수가 늘어날 때 시각 품질과 복잡도가 어떻게 변하는지 비교합니다.
k_values = [5, 10, 20]
comparison_items = [("Original", image)]
timing_rows = []

for k in k_values:
    # 같은 입력 이미지에 대해 K만 바꿔 K-Means를 반복 실행합니다.
    (result, palette), runtime = timed_call(kmeans_quantization, image, k)
    save_image_rgb(f"outputs/01_kmeans_k{k}.png", result)
    comparison_items.append((f"K-Means K={k} ({runtime:.3f}s)", result))
    timing_rows.append({"algorithm": "K-Means", "K": k, "runtime_sec": runtime, "palette_size": len(palette)})

show_images(comparison_items, cols=2, figsize=(11, 10), save_path="outputs/01_k_compare.png")
print_table(timing_rows)

## 비교 분석

- 색상 유지 품질: K-Means는 실제 이미지 색상 분포에 맞춰 대표색을 찾기 때문에 세 알고리즘 중 원본 느낌을 가장 잘 유지하는 편입니다.
- 경계 자연스러움: K-Means와 Median Cut은 색상 분포 기반이라 Posterization보다 계단 현상이 적습니다.
- 처리 속도: Posterization이 가장 빠르고, K-Means는 반복 최적화 때문에 중간, Median Cut은 구현 방식에 따라 중간 정도입니다.
- 사용자 관점: K가 작을수록 색칠은 쉽지만 원본 정보가 손실됩니다. 발표용 기본값은 보통 `K=10`이 균형적입니다.